In [1]:
import numpy as np
import pandas as pd
import torch
import os
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
COMBINED_EMB_PATH = "brain/filtered_visual_embeddings_combined.npy"
FILTERED_POI_PATH = "brain/filtered_poi_ids.npy"
FILTERED_PID_PATH = "brain/filtered_photo_ids.npy"

OUTPUT_DIR        = "brain/"
PCA_DIM = 34
# ICA-FCM params
K_MIN             = 5       # minimum clusters to try
K_MAX             = 20      # maximum clusters to try
FCM_M        = 1.3   # lower = sharper memberships (1.0=hard, 2.0=too fuzzy)
FCM_MAX_ITER = 150
FCM_TOL      = 1e-4

# ICA params
N_EMPIRES         = 5       # number of imperialists
ICA_MAX_ITER      = 50      # ICA iterations per K

print("Config ready.")

Config ready.


In [3]:
from sklearn.decomposition import PCA
import numpy as np

# Load a sample to check variance
sample = np.load("brain/filtered_visual_embeddings_combined.npy")
sample = sample[np.random.choice(len(sample), 10000, replace=False)]
sample = sample / (np.linalg.norm(sample, axis=1, keepdims=True) + 1e-8)

pca_test = PCA(n_components=100, random_state=42)
pca_test.fit(sample)

cumvar = np.cumsum(pca_test.explained_variance_ratio_)
for target in [0.80, 0.85, 0.90, 0.95, 0.99]:
    dims = np.argmax(cumvar >= target) + 1
    print(f"  {target*100:.0f}% variance → {dims} dims")

# Fix: explicit 99% check
print(f"\nFull cumvar at 100 dims: {cumvar[-1]:.4f}")
print(f"Cumvar at 34 dims      : {cumvar[33]:.4f}")
print(f"Cumvar at 50 dims      : {cumvar[49]:.4f}")

  80% variance → 9 dims
  85% variance → 10 dims
  90% variance → 11 dims
  95% variance → 35 dims
  99% variance → 1 dims

Full cumvar at 100 dims: 0.9800
Cumvar at 34 dims      : 0.9495
Cumvar at 50 dims      : 0.9607


In [4]:
print("Loading combined embeddings...")
combined_emb = np.load(COMBINED_EMB_PATH).astype(np.float32)
poi_ids      = np.load(FILTERED_POI_PATH)
photo_ids    = np.load(FILTERED_PID_PATH)

print(f"Combined embeddings : {combined_emb.shape}")

# Compute POI-level embeddings by averaging photos per POI
# Clustering happens at POI level, not photo level
print("Aggregating to POI level...")
unique_pois     = np.unique(poi_ids)
poi_embeddings  = np.zeros((len(unique_pois), combined_emb.shape[1]), dtype=np.float32)
poi_idx_map     = {poi: idx for idx, poi in enumerate(unique_pois)}

for poi in tqdm(unique_pois, desc="Averaging POI embeddings"):
    mask               = poi_ids == poi
    poi_embeddings[poi_idx_map[poi]] = combined_emb[mask].mean(axis=0)

print(f"POI-level embeddings: {poi_embeddings.shape}")

# L2 normalize
poi_embeddings = normalize(poi_embeddings, norm='l2', axis=1)

# PCA compression
print(f"Compressing {poi_embeddings.shape[1]}-dim → {PCA_DIM}-dim via PCA...")
pca            = PCA(n_components=PCA_DIM, random_state=42)
poi_emb_pca    = pca.fit_transform(poi_embeddings)
explained_var  = pca.explained_variance_ratio_.sum()

print(f"PCA explained variance : {explained_var*100:.1f}%")
print(f"Compressed shape       : {poi_emb_pca.shape}")

Loading combined embeddings...
Combined embeddings : (767210, 647)
Aggregating to POI level...


Averaging POI embeddings: 100%|████████████████████████████████████████████████| 169845/169845 [06:19<00:00, 447.92it/s]


POI-level embeddings: (169845, 647)
Compressing 647-dim → 34-dim via PCA...
PCA explained variance : 95.2%
Compressed shape       : (169845, 34)


In [5]:
from sklearn.cluster import KMeans

class FuzzyCMeans:
    """
    Fuzzy C-Means with K-Means++ initialization.
    K-Means++ gives smart centroid seeds — prevents degenerate convergence.
    """
    def __init__(self, n_clusters, m=2.0, max_iter=150, tol=1e-4):
        self.K        = n_clusters
        self.m        = m
        self.max_iter = max_iter
        self.tol      = tol

    def fit(self, X):
        N = len(X)

        # Step 1: K-Means++ initialization for smart centroid seeds
        km       = KMeans(n_clusters=self.K, init='k-means++',
                          n_init=3, max_iter=50, random_state=42)
        km.fit(X)
        centers  = km.cluster_centers_.copy()  # (K, D)

        # Step 2: Initialize membership from distances to K-Means++ centers
        dists = np.zeros((self.K, N))
        for k in range(self.K):
            diff     = X - centers[k]
            dists[k] = np.linalg.norm(diff, axis=1) + 1e-8

        exp  = 2 / (self.m - 1)
        U    = np.zeros((self.K, N))
        for k in range(self.K):
            ratio  = dists[k:k+1] / dists
            U[k]   = 1.0 / (ratio ** exp).sum(axis=0)

        # Step 3: FCM iterations
        for iteration in range(self.max_iter):
            U_old = U.copy()

            Um      = U ** self.m
            centers = (Um @ X) / Um.sum(axis=1, keepdims=True)

            dists = np.zeros((self.K, N))
            for k in range(self.K):
                diff     = X - centers[k]
                dists[k] = np.linalg.norm(diff, axis=1) + 1e-8

            U_new = np.zeros_like(U)
            for k in range(self.K):
                ratio      = dists[k:k+1] / dists
                U_new[k]   = 1.0 / (ratio ** exp).sum(axis=0)

            diff_norm = np.linalg.norm(U_new - U_old)
            U         = U_new

            if diff_norm < self.tol:
                print(f"    FCM converged at iteration {iteration+1}")
                break

        self.U       = U.T
        self.centers = centers
        self.labels  = U.argmax(axis=0)
        return self

    def predict(self, X):
        N     = len(X)
        dists = np.zeros((self.K, N))
        for k in range(self.K):
            diff     = X - self.centers[k]
            dists[k] = np.linalg.norm(diff, axis=1) + 1e-8
        exp   = 2 / (self.m - 1)
        U     = np.zeros((self.K, N))
        for k in range(self.K):
            ratio  = dists[k:k+1] / dists
            U[k]   = 1.0 / (ratio ** exp).sum(axis=0)
        return U.T

In [6]:
# Quick FCM test with m=1.3
test_fcm = FuzzyCMeans(n_clusters=8, m=1.3, max_iter=150, tol=1e-4)
test_fcm.fit(poi_emb_pca)
print("Sample memberships:")
for i in range(3):
    mem  = test_fcm.U[i]
    top2 = np.argsort(mem)[-2:][::-1]
    print(f"  POI {i}: {mem.round(3)} → primary {top2[0]} ({mem[top2[0]]:.1%}), secondary {top2[1]} ({mem[top2[1]]:.1%})")

    FCM converged at iteration 75
Sample memberships:
  POI 0: [0.026 0.002 0.436 0.002 0.005 0.507 0.002 0.021] → primary 5 (50.7%), secondary 2 (43.6%)
  POI 1: [0.125 0.004 0.54  0.003 0.008 0.263 0.002 0.055] → primary 2 (54.0%), secondary 5 (26.3%)
  POI 2: [0.067 0.002 0.468 0.002 0.004 0.436 0.001 0.02 ] → primary 2 (46.8%), secondary 5 (43.6%)


In [7]:
def ica_optimize_k(X, k_min, k_max, n_empires, ica_max_iter):
    """
    ICA treats each candidate K as a 'country'.
    Imperialists = best K values found so far.
    Colonies = other K values competing to improve.
    Assimilation + revolution drives search toward optimal K.
    Fitness = silhouette score (higher = better separated clusters).
    """
    # Initialize candidate K values as countries
    all_k      = list(range(k_min, k_max + 1))
    random.shuffle(all_k) if False else None
    np.random.shuffle(all_k := np.array(all_k))

    # Evaluate initial fitness for all K
    print("Evaluating initial K candidates...")
    fitness    = {}
    fcm_models = {}

    for k in tqdm(all_k, desc="Initial FCM runs"):
        fcm            = FuzzyCMeans(n_clusters=k, m=FCM_M,
                                     max_iter=FCM_MAX_ITER, tol=FCM_TOL)
        fcm.fit(X)
        # Silhouette on subsample for speed
        sample_idx     = np.random.choice(len(X), min(3000, len(X)), replace=False)
        sil            = silhouette_score(X[sample_idx], fcm.labels[sample_idx])
        db             = davies_bouldin_score(X[sample_idx], fcm.labels[sample_idx])
        fitness[k]     = sil - 0.1 * db   # combined score
        fcm_models[k]  = fcm
        print(f"  K={k:2d} | Silhouette: {sil:.4f} | DB: {db:.4f} | Fitness: {fitness[k]:.4f}")

    # Sort by fitness — top N_EMPIRES become imperialists
    sorted_k        = sorted(fitness, key=fitness.get, reverse=True)
    imperialists    = sorted_k[:n_empires]
    colonies        = sorted_k[n_empires:]

    print(f"\nImperialists (best K): {imperialists}")

    # ICA iterations — assimilation + revolution
    for iteration in range(ica_max_iter):
        # Assimilation: colonies move toward nearest imperialist
        for col in colonies:
            nearest_imp  = min(imperialists, key=lambda imp: abs(imp - col))
            # Move colony K toward imperialist K
            new_k        = col + int(0.5 * (nearest_imp - col))
            new_k        = max(k_min, min(k_max, new_k))

            if new_k not in fitness:
                fcm           = FuzzyCMeans(n_clusters=new_k, m=FCM_M,
                                            max_iter=FCM_MAX_ITER, tol=FCM_TOL)
                fcm.fit(X)
                sample_idx    = np.random.choice(len(X), min(3000, len(X)), replace=False)
                sil           = silhouette_score(X[sample_idx], fcm.labels[sample_idx])
                db            = davies_bouldin_score(X[sample_idx], fcm.labels[sample_idx])
                fitness[new_k]    = sil - 0.1 * db
                fcm_models[new_k] = fcm

        # Revolution: random perturbation of weakest colony
        weakest_col  = min(colonies, key=lambda c: fitness.get(c, -999))
        rev_k        = np.random.randint(k_min, k_max + 1)
        if rev_k not in fitness:
            fcm           = FuzzyCMeans(n_clusters=rev_k, m=FCM_M,
                                        max_iter=FCM_MAX_ITER, tol=FCM_TOL)
            fcm.fit(X)
            sample_idx    = np.random.choice(len(X), min(3000, len(X)), replace=False)
            sil           = silhouette_score(X[sample_idx], fcm.labels[sample_idx])
            db            = davies_bouldin_score(X[sample_idx], fcm.labels[sample_idx])
            fitness[rev_k]    = sil - 0.1 * db
            fcm_models[rev_k] = fcm

        # Update imperialists
        all_evaluated = sorted(fitness, key=fitness.get, reverse=True)
        imperialists  = all_evaluated[:n_empires]
        colonies      = all_evaluated[n_empires:]

    # Best K = top imperialist
    best_k     = imperialists[0]
    best_model = fcm_models[best_k]
    print(f"\nOptimal K = {best_k} | Best Fitness = {fitness[best_k]:.4f}")
    return best_k, best_model, fitness

import random
best_k, best_fcm, all_fitness = ica_optimize_k(
    poi_emb_pca, K_MIN, K_MAX, N_EMPIRES, ICA_MAX_ITER
)

Evaluating initial K candidates...


Initial FCM runs:   6%|███▉                                                           | 1/16 [09:09<2:17:19, 549.27s/it]

  K=14 | Silhouette: 0.0899 | DB: 2.3324 | Fitness: -0.1433


Initial FCM runs:  12%|███████▉                                                       | 2/16 [12:39<1:21:38, 349.89s/it]

    FCM converged at iteration 113
  K= 9 | Silhouette: 0.1062 | DB: 2.3186 | Fitness: -0.1257


...

  K=13 | Silhouette: 0.0919 | DB: 2.3588 | Fitness: -0.1440


Initial FCM runs:  94%|██████████████████████████████████████████████████████████▏   | 15/16 [2:01:17<06:20, 380.74s/it]

    FCM converged at iteration 75
  K= 8 | Silhouette: 0.1084 | DB: 2.2668 | Fitness: -0.1183


Initial FCM runs: 100%|██████████████████████████████████████████████████████████████| 16/16 [2:06:50<00:00, 475.67s/it]

  K=10 | Silhouette: 0.1024 | DB: 2.3276 | Fitness: -0.1304

Imperialists (best K): [np.int64(6), np.int64(8), np.int64(7), np.int64(9), np.int64(10)]

Optimal K = 6 | Best Fitness = -0.1097


In [8]:
ks      = sorted(all_fitness.keys())
scores  = [all_fitness[k] for k in ks]

plt.figure(figsize=(10, 4))
plt.plot(ks, scores, 'o-', color='steelblue')
plt.axvline(best_k, color='red', linestyle='--', label=f'Optimal K={best_k}')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Fitness (Silhouette - 0.1*DB)")
plt.title("ICA-FCM: Optimal K Search")
plt.legend()
plt.tight_layout()
plt.show()
plt.savefig(os.path.join(OUTPUT_DIR, "ica_fcm_k_search.png"), dpi=150)
plt.close()
print(f"Plot saved. Optimal K = {best_k}")

Plot saved. Optimal K = 6


In [9]:
"""
Each POI gets a K-dimensional membership vector.
e.g. [0.6, 0.1, 0.3, ...] = 60% cluster-0, 10% cluster-1, 30% cluster-2
This is the 'fuzzy probability' for Module C.
"""

print(f"Assigning fuzzy memberships with K={best_k}...")
poi_memberships = best_fcm.U   # (num_pois, K) — already computed during fit

print(f"Membership matrix shape : {poi_memberships.shape}")
print(f"Sample memberships (first 3 POIs):")
for i in range(3):
    mem = poi_memberships[i]
    top2 = np.argsort(mem)[-2:][::-1]
    print(f"  POI {unique_pois[i]}: {mem.round(3)} → primary cluster {top2[0]} ({mem[top2[0]]:.1%}), secondary {top2[1]} ({mem[top2[1]]:.1%})")

Assigning fuzzy memberships with K=6...
Membership matrix shape : (169845, 6)
Sample memberships (first 3 POIs):
  POI 0: [0.049 0.028 0.005 0.002 0.915 0.002] → primary cluster 4 (91.5%), secondary 0 (4.9%)
  POI 1: [0.253 0.108 0.013 0.004 0.619 0.003] → primary cluster 4 (61.9%), secondary 0 (25.3%)
  POI 2: [0.134 0.032 0.005 0.002 0.825 0.002] → primary cluster 4 (82.5%), secondary 0 (13.4%)


In [11]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("Checking cluster structure...")

# 1. Are distances uniform?
sample_idx = np.random.choice(len(poi_emb_pca), 500, replace=False)
sample     = poi_emb_pca[sample_idx]
dists      = np.linalg.norm(sample[:, None] - sample[None, :], axis=-1)
print(f"Distance stats — Min: {dists.min():.4f} | Max: {dists.max():.4f} | Std: {dists.std():.4f}")

# 2. Try hard K-Means — does it produce meaningful clusters?
km = KMeans(n_clusters=8, init='k-means++', n_init=5, random_state=42)
km.fit(poi_emb_pca)
sil = silhouette_score(poi_emb_pca[sample_idx], km.labels_[sample_idx])

print(f"K-Means Silhouette (K=6): {sil:.4f}")
print(f"Cluster sizes: {np.bincount(km.labels_)}")

Checking cluster structure...
Distance stats — Min: 0.0000 | Max: 1.7742 | Std: 0.2409
K-Means Silhouette (K=6): 0.1165
Cluster sizes: [17118 17996 23949 25341 20451 17800 19218 27972]


In [12]:
# Save membership matrix
np.save(os.path.join(OUTPUT_DIR, "poi_fuzzy_memberships.npy"), poi_memberships)

# Save POI → cluster index mapping
np.save(os.path.join(OUTPUT_DIR, "poi_cluster_ids.npy"),    unique_pois)
np.save(os.path.join(OUTPUT_DIR, "poi_hard_labels.npy"),    best_fcm.labels)
np.save(os.path.join(OUTPUT_DIR, "cluster_centers.npy"),    best_fcm.centers)

# Save as DataFrame for easy lookup in later notebooks
poi_membership_df = pd.DataFrame(
    poi_memberships,
    columns=[f"cluster_{i}" for i in range(best_k)]
)
poi_membership_df.insert(0, 'poi_id', unique_pois)
poi_membership_df.to_csv(os.path.join(OUTPUT_DIR, "poi_fuzzy_memberships.csv"), index=False)

print(f"Saved:")
print(f"  poi_fuzzy_memberships.npy : {poi_memberships.shape}")
print(f"  poi_fuzzy_memberships.csv : {poi_membership_df.shape}")
print(f"  cluster_centers.npy       : {best_fcm.centers.shape}")
print(f"  Optimal K                 : {best_k}")

Saved:
  poi_fuzzy_memberships.npy : (169845, 6)
  poi_fuzzy_memberships.csv : (169845, 7)
  cluster_centers.npy       : (6, 34)
  Optimal K                 : 6
